In [7]:
import numpy as np

NEURAL_PATH = "/home/maria/Science/data/hybrid_neural_responses_reduced.npy"
HUMAN_LABEL_PATH = "/home/maria/Science/data/image_labels.npy"

# Convention:
# 0 = inanimate
# 1 = animate

X = np.load(NEURAL_PATH)
y = np.load(HUMAN_LABEL_PATH, allow_pickle=True).item()["labels"]

print("Raw X shape:", X.shape)
print("Raw y shape:", y.shape)
print("Raw y dtype:", y.dtype)

# Fix labels if they were saved as an object containing one array
if y.shape == (1,):
    y = y.item()

y = np.asarray(y).astype(int).ravel()

print("Fixed y shape:", y.shape)
print("Unique labels:", np.unique(y, return_counts=True))

# Your X is neurons × images, so transpose to images × neurons
if X.shape[0] == y.shape[0]:
    X_images_by_neurons = X
elif X.shape[1] == y.shape[0]:
    X_images_by_neurons = X.T
else:
    raise ValueError(f"Cannot align X shape {X.shape} with y shape {y.shape}")

print("Aligned X shape:", X_images_by_neurons.shape)

# Ignore unlabeled images if any are marked -1
mask = y != -1
X_images_by_neurons = X_images_by_neurons[mask]
y = y[mask]

inanimate = X_images_by_neurons[y == 0]
animate = X_images_by_neurons[y == 1]

print()
print("Number inanimate:", len(inanimate))
print("Number animate:  ", len(animate))

inanimate_mean = inanimate.mean(axis=0)
animate_mean = animate.mean(axis=0)

inanimate_norm = np.linalg.norm(inanimate_mean)
animate_norm = np.linalg.norm(animate_mean)

print()
print("||mean inanimate||:", inanimate_norm)
print("||mean animate||:  ", animate_norm)
print()
print("animate - inanimate:", animate_norm - inanimate_norm)
print("animate / inanimate:", animate_norm / inanimate_norm)

if animate_norm > inanimate_norm:
    print("\nAnimate average neural activity vector has the larger norm.")
elif inanimate_norm > animate_norm:
    print("\nInanimate average neural activity vector has the larger norm.")
else:
    print("\nThe norms are exactly equal.")

Raw X shape: (39209, 118)
Raw y shape: (118,)
Raw y dtype: int64
Fixed y shape: (118,)
Unique labels: (array([0, 1]), array([62, 56]))
Aligned X shape: (118, 39209)

Number inanimate: 62
Number animate:   56

||mean inanimate||: 11.388015274114045
||mean animate||:   12.305603844159284

animate - inanimate: 0.9175885700452397
animate / inanimate: 1.080574933204647

Animate average neural activity vector has the larger norm.


In [12]:
import numpy as np

NEURAL_PATH = "/home/maria/Science/data/hybrid_neural_responses_reduced.npy"
HUMAN_LABEL_PATH = "/home/maria/Science/data/image_labels.npy"

# Convention:
# 0 = inanimate
# 1 = animate

N_PERMUTATIONS = 10_000
RANDOM_SEED = 42

X = np.load(NEURAL_PATH)
y = np.load(HUMAN_LABEL_PATH, allow_pickle=True).item()["labels"]

# Fix labels if saved as a nested object
if y.shape == (1,):
    y = y.item()

y = np.asarray(y).astype(int).ravel()

# Align X to images × neurons
if X.shape[0] == y.shape[0]:
    X = X
elif X.shape[1] == y.shape[0]:
    X = X.T
else:
    raise ValueError(f"Cannot align X shape {X.shape} with y shape {y.shape}")

# Remove unlabeled examples if any
mask = y != -1
X = X[mask]
y = y[mask]

print("Aligned X shape:", X.shape)
print("y shape:", y.shape)
print("Label counts:", dict(zip(*np.unique(y, return_counts=True))))

def norm_diff_stat(X, labels):
    """
    Positive means animate has larger mean-vector norm.
    """
    inanimate_mean = X[labels == 0].mean(axis=0)
    animate_mean = X[labels == 1].mean(axis=0)

    inanimate_norm = np.linalg.norm(inanimate_mean)
    animate_norm = np.linalg.norm(animate_mean)

    return animate_norm - inanimate_norm

observed_diff = norm_diff_stat(X, y)

rng = np.random.default_rng(RANDOM_SEED)
perm_diffs = np.empty(N_PERMUTATIONS)

for i in range(N_PERMUTATIONS):
    y_perm = rng.permutation(y)
    perm_diffs[i] = norm_diff_stat(X, y_perm)

# One-sided p-value: animate norm is larger
p_one_sided = (np.sum(perm_diffs >= observed_diff) + 1) / (N_PERMUTATIONS + 1)

# Two-sided p-value: either class has unusually larger norm
p_two_sided = (np.sum(np.abs(perm_diffs) >= abs(observed_diff)) + 1) / (N_PERMUTATIONS + 1)

print()
print("Observed animate - inanimate norm diff:", observed_diff)
print()
print("Permutation mean diff:", perm_diffs.mean())
print("Permutation std diff: ", perm_diffs.std(ddof=1))
print()
print("One-sided p-value, animate > inanimate:", p_one_sided)
print("Two-sided p-value:", p_two_sided)

print()
print("Permutation null 2.5%, 50%, 97.5% quantiles:")
print(np.quantile(perm_diffs, [0.025, 0.5, 0.975]))

if p_one_sided < 0.05:
    print("\nResult: animate mean-vector norm is significantly larger by permutation test.")
else:
    print("\nResult: not significant at alpha = 0.05.")

Aligned X shape: (118, 39209)
y shape: (118,)
Label counts: {np.int64(0): np.int64(62), np.int64(1): np.int64(56)}

Observed animate - inanimate norm diff: 0.9175885700452397

Permutation mean diff: 0.009723638186524636
Permutation std diff:  0.2532836331417207

One-sided p-value, animate > inanimate: 9.999000099990002e-05
Two-sided p-value: 0.00019998000199980003

Permutation null 2.5%, 50%, 97.5% quantiles:
[-0.48550935  0.01068286  0.4970305 ]

Result: animate mean-vector norm is significantly larger by permutation test.
